In [0]:
from pyspark.sql.functions import *

fact = spark.table("bankingdata.gold.fact_transaction")
dim_card = spark.table("bankingdata.gold.dim_card")

In [0]:
from pyspark.sql.functions import *

### KPI Data

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# window for top customer
w = Window.partitionBy("customer_id")

kpi_final = fact \
    .withColumn("cust_total_spend", sum("amount").over(w)) \
    .agg(
        sum("amount").alias("total_amount"),
        count("*").alias("total_transactions"),
        sum(when(col("is_high_value") == True, 1).otherwise(0)).alias("high_value_txn"),
        sum("amount").alias("branch_revenue"),
        max("cust_total_spend").alias("top_customer_spend"),
        avg("amount").alias("avg_txn_amount")
    )

# card txn count (separate logic as per your notebook)
kpi_card_count = fact.join(dim_card, fact.card_id == dim_card.CardID, "inner") \
                     .agg(count("*").alias("card_txn_count"))

# combine into ONE row
kpi_final = kpi_final.crossJoin(kpi_card_count)

In [0]:
# save delta table
kpi_final.write.mode("overwrite") \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bankingdata.businessinsights.kpi_summary")

### Graph Datasets

1. Transaction Trend

In [0]:
graph_trend = fact.withColumn("txn_date", to_date("event_time")) \
                  .groupBy("txn_date") \
                  .agg(sum("amount").alias("total_amount")) \
                  .orderBy("txn_date")

In [0]:
graph_trend.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bankingdata.businessinsights.graph_trend")

2. Revenue by Branch

In [0]:
graph_branch = fact.groupBy("branch_id") \
                   .agg(sum("amount").alias("total_amount"))

In [0]:
graph_branch.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bankingdata.businessinsights.graph_branch")

3. Card Type Distribution

In [0]:
graph_card = fact.join(dim_card, fact.card_id == dim_card.CardID) \
                 .groupBy("CardType") \
                 .agg(count("*").alias("txn_count"))

In [0]:
graph_card.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bankingdata.businessinsights.graph_card")

4. Merchant Category

In [0]:
graph_merchant = fact.groupBy("merchant_category") \
                     .agg(count("*").alias("txn_count"))

In [0]:
graph_merchant.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bankingdata.businessinsights.graph_merchant")

5. Top Customers

In [0]:
graph_top_customers = fact.groupBy("customer_id") \
                         .agg(sum("amount").alias("total_spend")) \
                         .orderBy(desc("total_spend")) \
                         .limit(10)

In [0]:
graph_top_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bankingdata.businessinsights.graph_top_customers")

6. High Value vs Normal

In [0]:
graph_high_value = fact.groupBy("is_high_value") \
                       .agg(count("*").alias("txn_count"))

In [0]:
graph_high_value.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bankingdata.businessinsights.graph_high_value")